# LoRA Adapter Stats (Tensor Sizes Only)

Load a saved LoRA checkpoint and print only the tensor shapes.

In [1]:
# Path to LoRA checkpoint: either .pt (state dict) or adapter dir with safetensors
LORA_PATH = "./sft_humaneval_output/lora_adapters/lora_state_dict.pt"
# Alternative: LORA_PATH = "./sft_humaneval_output/lora_adapters"  # for adapter_model.safetensors

In [2]:
import os

def load_lora_state(path):
    """Load LoRA state from .pt file or PEFT adapter dir."""
    if path.endswith(".pt"):
        return torch.load(path, map_location="cpu", weights_only=True)
    # Assume adapter dir with adapter_model.safetensors
    try:
        from safetensors.torch import load_file
        sf_path = os.path.join(path, "adapter_model.safetensors")
        return load_file(sf_path)
    except Exception as e:
        raise FileNotFoundError(f"Could not load from {path}: {e}") from e

In [3]:
import torch

lora_state = load_lora_state(LORA_PATH)

print("=== LoRA adapter tensor sizes ===")
for name, tensor in lora_state.items():
    print(f"  {name}: {tensor.shape}")

total_elements = sum(t.numel() for t in lora_state.values())
print(f"\n  Total tensors: {len(lora_state)}")
print(f"  Total elements: {total_elements:,}")

=== LoRA adapter tensor sizes ===
  base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight: torch.Size([16, 2048])
  base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight: torch.Size([2048, 16])
  base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight: torch.Size([16, 2048])
  base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight: torch.Size([256, 16])
  base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight: torch.Size([16, 2048])
  base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight: torch.Size([256, 16])
  base_model.model.model.layers.0.self_attn.o_proj.lora_A.default.weight: torch.Size([16, 2048])
  base_model.model.model.layers.0.self_attn.o_proj.lora_B.default.weight: torch.Size([2048, 16])
  base_model.model.model.layers.0.mlp.gate_proj.lora_A.default.weight: torch.Size([16, 2048])
  base_model.model.model.layers.0.mlp.gate_proj.lora_B.default.weight: torch.Size([11008, 16])
  b

## Parse and derive LoRA shapes (W, A, B)

**LoRA formula**: `output = W @ x + (B @ A) @ x` → effective update is `ΔW = B @ A` where:
- **A**: `[r, in_dim]` — projects input into rank-`r` space
- **B**: `[out_dim, r]` — projects from rank-`r` back to output
- **W**: `[out_dim, in_dim]` — original weight (frozen)

**How to derive from `lora_A` and `lora_B`**:
```python
r, in_dim  = lora_A.shape[0], lora_A.shape[1]   # A is [r, in]
out_dim    = lora_B.shape[0]                     # B is [out, r]
W_shape    = (out_dim, in_dim)
```

In [4]:
import re
from collections import defaultdict

def parse_lora_name(name):
    """Parse e.g. 'base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight'"""
    m = re.match(r".*\.layers\.(\d+)\.(self_attn|mlp)\.(\w+_proj)\.lora_(A|B)\.default\.weight", name)
    if m:
        return {"layer": int(m.group(1)), "block": m.group(2), "module": m.group(3), "lora": m.group(4)}
    return None

def derive_shapes(lora_state):
    """Parse state dict and group by module type with derived W, A, B shapes."""
    rows = {}  # module -> {W, A, B, params}
    for name, tensor in lora_state.items():
        p = parse_lora_name(name)
        if not p:
            continue
        key = (p["block"], p["module"])
        if key not in rows:
            rows[key] = {"A": None, "B": None}
        if p["lora"] == "A":
            rows[key]["A"] = tensor.shape
        else:
            rows[key]["B"] = tensor.shape

    # Derive W shape and param counts
    result = []
    for (block, module), shapes in sorted(rows.items(), key=lambda x: (x[0][0], x[0][1])):
        A, B = shapes["A"], shapes["B"]
        if A is None or B is None:
            continue
        r, in_dim = A[0], A[1]
        out_dim = B[0]
        assert B[1] == r, f"B.shape[1]={B[1]} != r={r}"
        W_shape = (out_dim, in_dim)
        params_A = A[0] * A[1]
        params_B = B[0] * B[1]
        result.append({
            "module": module,
            "W_shape": W_shape,
            "A_shape": tuple(A),
            "B_shape": tuple(B),
            "params": params_A + params_B,
        })
    return result

rows = derive_shapes(lora_state)

# Config inferred from shapes
q_row = next((x for x in rows if x["module"] == "q_proj"), None)
hidden = q_row["A_shape"][1] if q_row else 0
r = q_row["A_shape"][0] if q_row else 0

gate_row = next((x for x in rows if x["module"] == "gate_proj"), None)
intermediate = gate_row["B_shape"][0] if gate_row else 0

k_row = next((x for x in rows if x["module"] == "k_proj"), None)
kv_dim = k_row["B_shape"][0] if k_row else 0

num_layers = max((parse_lora_name(n)["layer"] for n in lora_state if parse_lora_name(n)), default=0) + 1

print("=== Derived config ===")
print(f"  hidden_size={hidden}, intermediate_size={intermediate}, r={r}, kv_dim={kv_dim}, num_layers={num_layers}")
print()

=== Derived config ===
  hidden_size=2048, intermediate_size=11008, r=16, kv_dim=256, num_layers=36



In [5]:
# Group by block (attention vs MLP) and print table
attn_rows = [r for r in rows if r["module"] in ("q_proj", "k_proj", "v_proj", "o_proj")]
mlp_rows = [r for r in rows if r["module"] in ("gate_proj", "up_proj", "down_proj")]

def fmt_shape(s):
    return str(list(s))

print("=== Attention projections ===")
print(f"{'Module':<10} {'W shape':<18} {'LoRA A shape':<18} {'LoRA B shape':<18} {'LoRA params'}")
print("-" * 80)
for r in attn_rows:
    params = f"{r['params']:,}"
    print(f"{r['module']:<10} {fmt_shape(r['W_shape']):<18} {fmt_shape(r['A_shape']):<18} {fmt_shape(r['B_shape']):<18} {params}")

print()
print("=== MLP projections ===")
print(f"{'Module':<10} {'W shape':<18} {'LoRA A shape':<18} {'LoRA B shape':<18} {'LoRA params'}")
print("-" * 80)
for r in mlp_rows:
    params = f"{r['params']:,}"
    print(f"{r['module']:<10} {fmt_shape(r['W_shape']):<18} {fmt_shape(r['A_shape']):<18} {fmt_shape(r['B_shape']):<18} {params}")

=== Attention projections ===
Module     W shape            LoRA A shape       LoRA B shape       LoRA params
--------------------------------------------------------------------------------
k_proj     [256, 2048]        [16, 2048]         [256, 16]          36,864
o_proj     [2048, 2048]       [16, 2048]         [2048, 16]         65,536
q_proj     [2048, 2048]       [16, 2048]         [2048, 16]         65,536
v_proj     [256, 2048]        [16, 2048]         [256, 16]          36,864

=== MLP projections ===
Module     W shape            LoRA A shape       LoRA B shape       LoRA params
--------------------------------------------------------------------------------
down_proj  [2048, 11008]      [16, 11008]        [2048, 16]         208,896
gate_proj  [11008, 2048]      [16, 2048]         [11008, 16]        208,896
up_proj    [11008, 2048]      [16, 2048]         [11008, 16]        208,896
